# Lab 4.4 &mdash; Ask Your App What It Just Did

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- Trace a payment-exception agent with one line of LangChain callback
- Ask, in English, which tools it called and how many model round-trips it took
- Turn those questions into behavioural tests that run against real traces
- Catch a regression the unit tests cannot see

> **How this lab works.** You write real LangChain and MCP code. Fill every `BLANK`, then run
> the **Self-check** cell under each section &mdash; those assert on the *objects you built*
> (a `@tool`, an argument schema, a `ToolMessage`, an `mcp.types.Tool`), so they are
> deterministic and do not depend on the model. Cells marked **Run it for real** put your code
> in front of the sandbox model; that is the part worth watching. The score line is feedback,
> not a grade.

> **The app is the point.** Labs 4.1&ndash;4.3 pointed agents at other people's systems.
> Here the system is *yours*, and MCP is how you interrogate it.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-4-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model can reason before it answers, and that reasoning is billed as completion
# tokens. It is off here because tool selection is a short decision and you will make a lot
# of them today. Pass think=True to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 4 labs -- the same payment exceptions as Day 1,
# now reached through tools the model chooses, and then through tools you did not write.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

You have an agent in front of a payments desk. It works &mdash; the answers look right.

Now the questions that actually matter in production:

- Did it **consult the policy**, or did it recall something plausible from training?
- How many **model round-trips** did that answer cost?
- After I reworded the system prompt, is it still calling the same tools?
- Which step is slow?

None of that is in the answer. All of it is in the trace. Langfuse already has it, and its MCP
server turns those questions into things you can ask in English &mdash; and then, once you know the
shape of the answer, into **assertions you can run in CI**.

That last step is why this lab is graded. A question you can ask is useful; a question you can
*fail a build on* is a test.

## Section 1 &mdash; The app, traced

The agent below is deliberately ordinary: two tools over the module's ledger, one system prompt.
The only unusual line is `CallbackHandler()`, which is all LangChain needs to emit a trace.

`run_name` matters more than it looks &mdash; it is the handle you will use to find this run again
among everyone else's.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langfuse import get_client
from langfuse.langchain import CallbackHandler

import re as _re, socket as _socket
# Your sandbox name, so your run is findable on a project the whole room shares.
WHOAMI   = (_re.search(r"(u\d+)", _socket.gethostname()) or _re.match(r"(x)", "x")).group(1)
RUN_NAME = f"triage-{WHOAMI}-{int(time.time())}"     # yours, findable, unique


@tool
def lookup_payment(ref: str) -> str:
    """Fetch one payment from the ledger by reference, e.g. PMT-1003."""
    return json.dumps(LEDGER.get(ref, {"error": "no such payment"}))


@tool
def policy_for(reason_code: str) -> str:
    """Return the written policy rule for a payment's reason code."""
    return POLICY.get(reason_code, "no rule on file")


def build_app(system_prompt: str):
    return create_agent(model=get_llm(), tools=[lookup_payment, policy_for],
                        system_prompt=system_prompt)


CAREFUL = ("You triage payment exceptions. Always look the payment up, then look up the policy "
           "rule for its reason code before answering. Quote the rule. Be brief.")


def run_traced(app, question: str, run_name: str):
    """Invoke the app, tracing it, and return (answer, window).

    The window is the point. Everyone in the room writes to this project, and you
    will run this agent more than once yourself -- so a question like "did it call
    policy_for" is meaningless unless it is asked about ONE run. Bracketing the
    invoke with timestamps is the whole trick.
    """
    t0 = time.time() - 2
    out = app.invoke({"messages": [("user", question)]},
                     config={"callbacks": [CallbackHandler()], "run_name": run_name})
    get_client().flush()                    # traces are batched; push them before we go looking
    return out["messages"][-1].content, (t0, time.time() + 2)


if llm_ready():
    careful = guard(lambda: run_traced(build_app(CAREFUL), "Triage PMT-1003.", RUN_NAME), (None, None))
    answer, CAREFUL_WINDOW = careful
    print("run name :", RUN_NAME)
    print("answer   :", str(answer)[:220])
else:
    CAREFUL_WINDOW = None
    print("Run it for real needs the model. See the setup cell.")

## Section 2 &mdash; What counts as evidence

Before asking Langfuse anything, decide what a *good* run looks like. That is the part people skip,
and it is the whole difference between a dashboard and a test.

For this agent, one claim matters more than the rest: **the answer quoted policy because it read
policy**, not because the model remembered something similar. There is exactly one observation name
that proves it.

In [ ]:
# A real queryMetrics response, captured from this project. No network needed.
BY_NAME = {"data": [
    {"name": "triage-u31-1788996887", "count_count": 1},
    {"name": "ChatOpenAI",            "count_count": 3},
    {"name": "tools",                 "count_count": 2},
    {"name": "lookup_payment",        "count_count": 1},
    {"name": "policy_for",            "count_count": 1},
]}


def times_called(rows: dict, name: str) -> int:
    """How many observations with this name are in a queryMetrics-by-name response."""
    return sum(r["count_count"] for r in rows["data"] if r["name"] == name)


def consulted_policy(rows: dict) -> bool:
    """True when the run actually READ the policy rather than recalling one."""
    return times_called(rows, "policy_for") > 0


def model_round_trips(rows: dict) -> int:
    """How many times the agent went back to the model. Each one costs money and latency."""
    return times_called(rows, "ChatOpenAI")

In [ ]:
# ---- Self-check: your assertions, against captured traces. No model, no network ----
LAZY = {"data": [{"name": "triage-u31-lazy", "count_count": 1},
                 {"name": "ChatOpenAI", "count_count": 1},
                 {"name": "lookup_payment", "count_count": 1}]}      # never opened the policy

check("a run that called policy_for counts as having consulted policy",
      lambda: consulted_policy(BY_NAME) is True)
check("a run that skipped it does not",
      lambda: consulted_policy(LAZY) is False,
      "this agent answered anyway -- the trace is the only place that shows it guessed")
check("round-trips are counted from the model observations",
      lambda: model_round_trips(BY_NAME) == 3)
check("and the cheap run is visibly cheaper",
      lambda: model_round_trips(LAZY) < model_round_trips(BY_NAME))
check("times_called does not confuse the run name with a tool",
      lambda: times_called(BY_NAME, "lookup_payment") == 1)
score()

## Run it for real &mdash; ask Langfuse about your own run

The bridge below is given to you. It is the same shape as Lab 4.2's config, in Python instead of
JSON: post to the MCP endpoint, hand back the text. Nothing here is new &mdash; the interesting part
is the question you ask with it.

⚠️ **One trap worth knowing.** `listObservations` will happily return the whole project's recent
history and yours may not be on the first page &mdash; it looks exactly like your trace never
arrived. **Ask `queryMetrics` with a `name` dimension instead.** That is how these were found.

In [ ]:
import base64, urllib.request

LF_URL  = os.environ.get("LANGFUSE_HOST", "").rstrip("/") + "/api/public/mcp"
LF_AUTH = base64.b64encode(f"{os.environ.get('LANGFUSE_PUBLIC_KEY','')}:"
                           f"{os.environ.get('LANGFUSE_SECRET_KEY','')}".encode()).decode()
_sess = {"id": None}


def langfuse_ready() -> bool:
    return all(os.environ.get(k) for k in ("LANGFUSE_HOST", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"))


def mcp(method: str, params: dict = None) -> dict:
    body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}}).encode()
    req = urllib.request.Request(LF_URL, data=body, method="POST")
    req.add_header("Content-Type", "application/json")
    req.add_header("Accept", "application/json, text/event-stream")
    req.add_header("Authorization", "Basic " + LF_AUTH)
    if _sess["id"]:
        req.add_header("Mcp-Session-Id", _sess["id"])
    with urllib.request.urlopen(req, timeout=120) as r:
        raw, sid = r.read().decode(), r.headers.get("mcp-session-id")
    if sid:
        _sess["id"] = sid
    for line in raw.splitlines():
        if line.startswith("data:"):
            raw = line[5:].strip()
            break
    return json.loads(raw)


def names_in(window) -> dict:
    """queryMetrics grouped by name, for ONE run's window -- the anatomy of what it did."""
    start, end = window
    mcp("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                       "clientInfo": {"name": "lab-4-4", "version": "1.0"}})
    env = mcp("tools/call", {"name": "queryMetrics", "arguments": {
        "view": "observations",
        "dimensions": [{"field": "name"}],
        "metrics": [{"measure": "count", "aggregation": "count"}],
        "filters": [],
        "fromTimestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(start)),
        "toTimestamp":   time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(end)),
    }})
    if "error" in env:
        return {"data": [], "error": str(env["error"])[:200]}
    text = "".join(c.get("text", "") for c in env.get("result", {}).get("content", []))
    return json.loads(text) if text.strip().startswith("{") else {"data": []}


if langfuse_ready() and llm_ready() and CAREFUL_WINDOW:
    time.sleep(6)                                   # ingestion is not instant
    rows = guard(lambda: names_in(CAREFUL_WINDOW), {"data": []})
    print("everything THIS run did:\n")
    for r in sorted(rows.get("data", []), key=lambda r: -r["count_count"])[:10]:
        print(f"  {r['count_count']:>3}  {r['name']}")
    print("\n--- your run, judged ---")
    print("  consulted policy :", consulted_policy(rows))
    print("  model round-trips:", model_round_trips(rows))
else:
    print("Needs both the model and Langfuse. See the setup cell.")

### The regression this catches

Now change one word in the prompt and watch the trace change while the *answer* does not.

In [ ]:
# The change under test: same prompt, same question, but this build of the app was
# never given the policy tool. It will still answer. Confidently.
def build_lazy_app():
    return create_agent(model=get_llm(), tools=[lookup_payment],
                        system_prompt=CAREFUL)


if llm_ready() and langfuse_ready():
    lazy_run = f"triage-{WHOAMI}-lazy-{int(time.time())}"
    lazy_answer, lazy_window = guard(
        lambda: run_traced(build_lazy_app(), "Triage PMT-1003.", lazy_run), (None, None))
    print("answer:", str(lazy_answer)[:220])
    print("\nReads well, doesn't it. Now the trace for THAT run alone:\n")
    time.sleep(6)
    after = guard(lambda: names_in(lazy_window), {"data": []}) if lazy_window else {"data": []}
    for r in sorted(after.get("data", []), key=lambda r: -r["count_count"])[:8]:
        print(f"  {r['count_count']:>3}  {r['name']}")
    print("\n  consulted policy :", consulted_policy(after))
    print("  round-trips      :", model_round_trips(after))
    print("\npolicy_for is absent, so whatever rule it quoted, it did not read one.")
    print("The answer never said so. Only the trace does.")
else:
    print("skipped - needs the model and Langfuse")

## Prompts worth keeping &mdash; and they double as tests

Paste these into the `opencode` agent you configured in Lab 4.2, which already has the Langfuse
server. Each is a question a developer asks about their own app, and each has an assertion hiding
inside it.

**Start every metrics prompt with *&ldquo;call getMetricsSchema first&rdquo;***, and add
*&ldquo;filter to environment = &lt;yours&gt;&rdquo;* when the project is busy.

### Did it behave?

```
Break observations down by name for the last 15 minutes. Did anything named policy_for run?
If not, say so plainly.
```
*The test: **an agent that quotes policy must have read policy.** Nothing in the answer text
tells you this. Run it after any prompt change.*

```
For the last 15 minutes, how many observations are named ChatOpenAI, and how many are tools?
Is that ratio what you would expect for an agent that calls two tools once each?
```
*The test: **round-trip count.** A jump here is a prompt that has started dithering, and it costs
money on every single run.*

### Is it still fast?

```
Call getMetricsSchema first. Then give me average and maximum latency by observation name for
the last hour, sorted by average. Which step dominates?
```
*The test: **a step that got slower.** Usually a tool doing more work than it used to.*

```
Compare the last 15 minutes against the hour before it: observation count, and average
latency by name. Has anything changed shape?
```
*The before-and-after you run around a deploy.*

### What actually happened in one run?

```
Find the observations whose name starts with "triage-" from the last 30 minutes, take the most
recent, and tell me every step it ran in order with its latency.
```
*The one to reach for when a single request behaved oddly. Otherwise: opening a trace and
scrolling.*

```
Across the last hour, which tool names appear most often? Is there a tool that never appears
at all?
```
*The test: **a tool nothing ever selects.** That is a description problem, and Module 4 is where
you fix it &mdash; but you cannot fix what you cannot see.*

## What this bought your app

The agent's answer was the same in both runs. **The trace was not.** That gap is the whole reason
this lab exists:

| the question | where the answer lives |
|---|---|
| is the answer right? | the response &mdash; and a human reading it |
| did it get there honestly? | the trace |
| did it get more expensive? | the trace |
| did my prompt change break something? | the trace |
| which step is slow? | the trace |

Unit tests cannot see any of that, because none of it is in the return value. Wiring Langfuse in
over MCP turns all of it into questions you can ask in English &mdash; and the useful ones you keep
and run every time you change the prompt.

## Your turn

- Tighten `LAZY_PROMPT` until `consulted_policy` goes green again. That loop is prompt engineering
  with a pass/fail.
- Add a third tool the agent rarely needs, run it ten times, and find out whether it is ever chosen.
- Write one more assertion for something you would want to fail a build on, and say out loud what
  it would have caught.